In [10]:
import os, sys, json, random, hashlib, subprocess, shutil
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')
!pip install -q kagglehub roboflow

import tensorflow as tf
import keras
import kagglehub

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = '/content/drive/MyDrive/Agrilens_pro'
for d in ('outputs', 'models', 'data', 'code'):
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)

gpus = tf.config.list_physical_devices('GPU')
env = {"python": sys.version.split()[0], "tensorflow": tf.__version__, "keras": keras.__version__,
       "numpy": np.__version__, "gpu": [g.name for g in gpus], "seed": SEED}
json.dump(env, open(f'{BASE_DIR}/outputs/env_info.json', 'w'), indent=2)
print(json.dumps(env, indent=2))
if not gpus:
    print("NOTE: no GPU attached (fine until training starts).")

Mounted at /content/drive
{
  "python": "3.13.16",
  "tensorflow": "2.21.0",
  "keras": "3.13.2",
  "numpy": "2.4.6",
  "gpu": [
    "/physical_device:GPU:0"
  ],
  "seed": 42
}


In [11]:
%%writefile /content/drive/MyDrive/Agrilens_pro/code/agrilens_utils.py
import os, json, hashlib, subprocess, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub

SEED, IMG = 42, 224
BASE_DIR = "/content/drive/MyDrive/Agrilens_pro"
OUT = BASE_DIR + "/outputs"
MODELS = BASE_DIR + "/models"
PD_DIR = "/content/PlantDoc"
CROPS = "/content/fieldplant_crops"
IMG_EXT = (".jpg", ".jpeg", ".png")

SPECIES = {
    "Apple Scab Leaf": "apple", "Apple leaf": "apple", "Apple rust leaf": "apple",
    "Bell_pepper leaf": "bell_pepper", "Bell_pepper leaf spot": "bell_pepper",
    "Blueberry leaf": "blueberry", "Cherry leaf": "cherry",
    "Corn Gray leaf spot": "corn", "Corn leaf blight": "corn", "Corn rust leaf": "corn",
    "Peach leaf": "peach", "Potato leaf early blight": "potato",
    "Potato leaf late blight": "potato", "Raspberry leaf": "raspberry",
    "Soyabean leaf": "soybean", "Squash Powdery mildew leaf": "squash",
    "Strawberry leaf": "strawberry", "grape leaf": "grape", "grape leaf black rot": "grape",
}
TOMATO_TO_CLASS = {
    "Tomato Early blight leaf": "early_blight", "Tomato Septoria leaf spot": "septoria_leaf_spot",
    "Tomato leaf bacterial spot": "bacterial_spot", "Tomato leaf late blight": "late_blight",
    "Tomato leaf mosaic virus": "mosaic_virus", "Tomato leaf yellow virus": "yellow_leaf_curl_virus",
    "Tomato mold leaf": "leaf_mold",
}
HOLDOUT = {"apple", "grape", "corn"}
EXPECTED = {
    "olazo_splits": {"train": 14193, "val": 1774, "cal": 1775, "test": 4437},
    "plantdoc_roles": {"train": 1302, "holdout": 779, "cal": 162, "val": 162, "ext_test": 161,
                       "dropped_dup_of_test": 8, "dropped_dup_in_holdout": 3, "dropped_dup_in_train": 1},
    "gate_rows": 22675,
}


def md5_file(p):
    with open(p, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def fingerprint(df, cols):
    lines = sorted("|".join(str(v) for v in row) for row in df[cols].itertuples(index=False, name=None))
    return hashlib.md5("\n".join(lines).encode()).hexdigest()


def backup_existing(path):
    if os.path.exists(path):
        os.replace(path, path + ".bak")


def get_roots():
    roots = {
        "olazo": kagglehub.dataset_download("luisolazo/tomato-diseases"),
        "emmarex": kagglehub.dataset_download("emmarex/plantdisease"),
        "intel": kagglehub.dataset_download("puneet6060/intel-image-classification"),
        "natural": kagglehub.dataset_download("prasunroy/natural-images"),
        "plantdoc": PD_DIR,
        "fieldplant": CROPS,
    }
    if not os.path.exists(PD_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/pratikkayal/PlantDoc-Dataset.git", PD_DIR], check=True)
    zp = BASE_DIR + "/data/fieldplant_crops.zip"
    if not os.path.exists(CROPS) and os.path.exists(zp):
        shutil.unpack_archive(zp, CROPS)
    return roots


def _load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG), method="bilinear")
    return tf.cast(img, tf.float32), tf.cast(label, tf.float32)


def make_ds(df, train=False, bs=32):
    ds = tf.data.Dataset.from_tensor_slices((df["file"].tolist(), df["label"].astype("float32").values))
    if train:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE).batch(bs).prefetch(tf.data.AUTOTUNE)


def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def gate_group(r):
    if r["source"] == "fieldplant":
        return "tomato_field_fp" if r["label"] == 1 else "cassava_fp"
    if r["label"] == 1:
        return "tomato_" + r["style"]
    if r["split"] == "holdout":
        return "unseen_" + r["group"]
    if r["style"] == "field":
        return "field_other_species"
    if r["group"] in ("potato", "bell_pepper"):
        return r["group"] + "_lab"
    return r["group"]


def gate_report(d, t):
    rows = []
    for g, x in d.groupby("grp"):
        n, k = len(x), int((x["p"] >= t).sum())
        lo, hi = wilson(k, n)
        rows.append((g, n, k, round(k / n, 3), round(max(lo, 0.0), 3), round(hi, 3)))
    return pd.DataFrame(rows, columns=["group", "n", "accepted", "rate", "ci_lo", "ci_hi"])


def evaluate_gate(model_path, G, tag, save_threshold=False):
    m = tf.keras.models.load_model(model_path)
    parts = []
    for sp in ("cal", "test", "ext_test", "holdout"):
        d = G[G["split"] == sp].copy()
        if len(d):
            d["p"] = m.predict(make_ds(d, bs=64), verbose=0).ravel()
            parts.append(d)
    P = pd.concat(parts, ignore_index=True)
    P["grp"] = P.apply(gate_group, axis=1)
    cal = P[P["split"] == "cal"]
    grid = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98]
    tab = pd.DataFrame({t: {g: (x["p"] >= t).mean() for g, x in cal.groupby("grp")} for t in grid}).T.round(3)
    neg_cols = [c for c in tab.columns if not c.startswith("tomato")]
    ok = tab[(tab[neg_cols] <= 0.05).all(axis=1)]
    thr = float(ok.index[0]) if len(ok) else grid[-1]
    print("Cal split accept rates (tomato = recall, others = false accepts)")
    print(tab.to_string())
    print("\nChosen threshold:", thr, "" if len(ok) else "(WARNING: no threshold gave <=5% false accepts everywhere)")
    for name in ("test", "ext_test", "holdout"):
        print("\n--- " + name + " ---")
        print(gate_report(P[P["split"] == name], thr).to_string(index=False))
    P.drop(columns=["file"], errors="ignore").to_csv(OUT + "/gate_predictions_" + tag + ".csv", index=False)
    if save_threshold:
        json.dump({"threshold": thr, "rule": "accept if p_tomato_leaf >= threshold",
                   "chosen_on": "cal split, every negative group <=5% false accepts"},
                  open(MODELS + "/gate_threshold.json", "w"), indent=2)
    return P, thr

Writing /content/drive/MyDrive/Agrilens_pro/code/agrilens_utils.py


In [12]:
import sys, importlib
sys.path.insert(0, f"{BASE_DIR}/code")
import agrilens_utils
importlib.reload(agrilens_utils)
from agrilens_utils import *
print("Utilities loaded:", all(n in globals() for n in ("get_roots", "make_ds", "evaluate_gate", "fingerprint")))

Utilities loaded: True


In [13]:
olazo = kagglehub.dataset_download("luisolazo/tomato-diseases")

def audit_split(root):
    files = [(c, os.path.join(root, c, fn)) for c in sorted(os.listdir(root))
             for fn in sorted(os.listdir(os.path.join(root, c)))]
    by_hash, bad = defaultdict(list), []
    for cls, p in tqdm(files, desc=os.path.basename(root)):
        try:
            with Image.open(p) as im:
                im.verify()
            by_hash[md5_file(p)].append((cls, p))
        except Exception:
            bad.append(p)
    return files, by_hash, bad

tr_files, tr_hash, tr_bad = audit_split(f"{olazo}/train")
te_files, te_hash, te_bad = audit_split(f"{olazo}/test")
assert (len(tr_files), len(te_files)) == (17753, 4440), "Olazo version differs from the audited one"
assert not tr_bad and not te_bad, "corrupt files found"
print("Unique train/test:", len(tr_hash), len(te_hash),
      "| train/test duplicates:", len(set(tr_hash) & set(te_hash)),
      "| label conflicts:", sum(len({c for c, _ in v}) > 1 for v in tr_hash.values()))

rel = lambda p: os.path.relpath(p, olazo)
test_rows, train_rows, excluded = [], [], []
for h, items in te_hash.items():
    test_rows.append((rel(items[0][1]), items[0][0], "test", h))
    excluded += [(rel(q), c, "duplicate_within_test", h) for c, q in items[1:]]
for h, items in tr_hash.items():
    if h in te_hash:
        excluded += [(rel(q), c, "also_in_test", h) for c, q in items]
        continue
    train_rows.append((rel(items[0][1]), items[0][0], h))
    excluded += [(rel(q), c, "duplicate_within_train", h) for c, q in items[1:]]

tdf = pd.DataFrame(train_rows, columns=["path", "label", "md5"])
tr, tmp = train_test_split(tdf, test_size=0.20, stratify=tdf["label"], random_state=SEED)
va, ca = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=SEED)
test_df = pd.DataFrame(test_rows, columns=["path", "label", "split", "md5"])
manifest = pd.concat([tr.assign(split="train"), va.assign(split="val"), ca.assign(split="cal"), test_df],
                     ignore_index=True)[["path", "label", "split", "md5"]]

sets = {s: set(g["md5"]) for s, g in manifest.groupby("split")}
assert all(not (sets[a] & sets[b]) for a in sets for b in sets if a < b), "split overlap"
class_names = sorted(manifest["label"].unique())
assert class_names == sorted(os.listdir(f"{olazo}/train"))
assert manifest["split"].value_counts().to_dict() == EXPECTED["olazo_splits"], "splits differ from the frozen ones"

manifest.to_csv(f"{OUT}/splits_v1.csv", index=False)
pd.DataFrame(excluded, columns=["path", "label", "reason", "md5"]).to_csv(f"{OUT}/excluded_v1.csv", index=False)
json.dump({str(i): c for i, c in enumerate(class_names)}, open(f"{MODELS}/class_names.json", "w"), indent=2)
print(pd.crosstab(manifest["label"], manifest["split"])[["train", "val", "cal", "test"]])

Using Colab cache for faster access to the 'tomato-diseases' dataset.


test: 100%|██████████| 4440/4440 [00:35<00:00, 126.74it/s]


Unique train/test: 17746 4437 | train/test duplicates: 4 | label conflicts: 0
split                   train  val  cal  test
label                                        
bacterial_spot           1343  168  168   444
early_blight             1741  218  218   444
healthy                  1502  188  188   443
late_blight              1669  209  209   442
leaf_mold                1168  146  146   444
mosaic_virus              958  120  119   444
septoria_leaf_spot       1062  132  133   444
target_spot               888  111  111   444
twospotted_spider_mite    986  123  123   444
yellow_leaf_curl_virus   2876  359  360   444


In [14]:
if not os.path.exists(PD_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/pratikkayal/PlantDoc-Dataset.git", PD_DIR], check=True)

rows, bad = [], []
for split in ("train", "test"):
    for cls in sorted(os.listdir(f"{PD_DIR}/{split}")):
        cdir = f"{PD_DIR}/{split}/{cls}"
        if not os.path.isdir(cdir):
            continue
        for fn in sorted(os.listdir(cdir)):
            p = os.path.join(cdir, fn)
            try:
                assert fn.lower().endswith(IMG_EXT)
                with Image.open(p) as im:
                    im.verify()
                rows.append((split, cls, os.path.relpath(p, PD_DIR), md5_file(p)))
            except Exception:
                bad.append(p)
assert not bad, f"unreadable files: {bad[:3]}"
pdf = pd.DataFrame(rows, columns=["split", "cls", "path", "md5"])
pdf["tomato"] = pdf["cls"].str.lower().str.startswith("tomato")
assert set(pdf.loc[~pdf["tomato"], "cls"]) == set(SPECIES)
assert set(pdf.loc[pdf["tomato"], "cls"]) == set(TOMATO_TO_CLASS) | {"Tomato leaf", "Tomato two spotted spider mites leaf"}
pdf["species"] = pdf["cls"].map(SPECIES).fillna("tomato")
pdf["classifier_label"] = pdf["cls"].map(TOMATO_TO_CLASS)

pdf["role"] = ""
test_hashes = set(pdf.loc[pdf["split"] == "test", "md5"])
is_train = pdf["split"] == "train"
pdf.loc[~is_train, "role"] = "ext_test"
pdf.loc[is_train & pdf["md5"].isin(test_hashes), "role"] = "dropped_dup_of_test"
pdf.loc[is_train & (pdf["role"] == "") & pdf.duplicated("md5", keep="first"), "role"] = "dropped_dup_in_train"
ho = pdf["species"].isin(HOLDOUT)
pdf.loc[ho, "role"] = "holdout"
sub = pdf[ho]
pdf.loc[sub[sub.duplicated("md5", keep="first")].index, "role"] = "dropped_dup_in_holdout"

rng = np.random.RandomState(SEED)
for cls, idx in pdf[pdf["role"] == ""].groupby("cls").groups.items():
    idx = list(idx)
    rng.shuffle(idx)
    k = int(round(0.10 * len(idx))) if len(idx) >= 10 else 0
    pdf.loc[idx[:k], "role"] = "val"
    pdf.loc[idx[k:2 * k], "role"] = "cal"
    pdf.loc[idx[2 * k:], "role"] = "train"

nun = pdf.groupby("md5")["cls"].nunique()
pdf["label_conflict"] = pdf["md5"].isin(set(nun[nun > 1].index))
pdf.loc[pdf["label_conflict"], "classifier_label"] = np.nan

keep_roles = ["train", "val", "cal", "ext_test", "holdout"]
pools = {r: set(g["md5"]) for r, g in pdf[pdf["role"].isin(keep_roles)].groupby("role")}
assert all(not (pools[a] & pools[b]) for a in pools for b in pools if a < b), "PlantDoc overlap"
assert not (set(pdf["md5"]) & set(pd.read_csv(f"{OUT}/splits_v1.csv")["md5"])), "PlantDoc overlaps Olazo"
assert pdf["role"].value_counts().to_dict() == EXPECTED["plantdoc_roles"], "roles differ from the frozen ones"
pdf.to_csv(f"{OUT}/plantdoc_manifest_v1.csv", index=False)
print(pdf["role"].value_counts().to_dict())

{'train': 1302, 'holdout': 779, 'cal': 162, 'val': 162, 'ext_test': 161, 'dropped_dup_of_test': 8, 'dropped_dup_in_holdout': 3, 'dropped_dup_in_train': 1}


In [15]:
roots = get_roots()
INTEL = {"mountain", "street", "buildings", "sea", "forest", "glacier"}
NATURAL = {"airplane", "car", "cat", "dog", "flower", "fruit", "motorbike", "person"}

rows = []
for source in ("emmarex", "intel", "natural"):
    paths = sorted(os.path.join(dp, fn) for dp, _, fns in os.walk(roots[source])
                   for fn in fns if fn.lower().endswith(IMG_EXT))
    for p in tqdm(paths, desc=source):
        cls = os.path.basename(os.path.dirname(p))
        if source == "emmarex":
            c = cls.lower()
            sp = "bell_pepper" if c.startswith("pepper") else "potato" if c.startswith("potato") else None
            if sp is None:
                continue
            group, style = sp, "lab"
        elif source == "intel":
            if cls not in INTEL:
                continue
            group, style = "scene", "scene"
        else:
            if cls not in NATURAL:
                continue
            group, style = "object", "scene"
        try:
            with Image.open(p) as im:
                im.verify()
            rows.append((source, os.path.relpath(p, roots[source]), cls, group, style, md5_file(p)))
        except Exception:
            pass

neg_all = pd.DataFrame(rows, columns=["source", "rel_path", "cls", "group", "style", "md5"])
neg_all = neg_all.drop_duplicates("md5", keep="first").reset_index(drop=True)
neg_all.to_csv(f"{OUT}/negatives_index_v1.csv", index=False)

def cap(df, total):
    per = max(1, total // df["cls"].nunique())
    return pd.concat([g.sample(min(len(g), per), random_state=SEED) for _, g in df.groupby("cls")])

neg = pd.concat([neg_all[neg_all["source"] == "emmarex"],
                 cap(neg_all[neg_all["source"] == "intel"], 1000),
                 cap(neg_all[neg_all["source"] == "natural"], 1000)])
neg["split"] = ""
rng = np.random.RandomState(SEED)
for _, idx in neg.groupby(["source", "cls"]).groups.items():
    idx = list(idx)
    rng.shuffle(idx)
    k = int(round(0.10 * len(idx)))
    neg.loc[idx[:k], "split"] = "val"
    neg.loc[idx[k:2 * k], "split"] = "cal"
    neg.loc[idx[2 * k:3 * k], "split"] = "test"
    neg.loc[idx[3 * k:], "split"] = "train"

ol = pd.read_csv(f"{OUT}/splits_v1.csv")
frac = 5500 / (ol["split"] == "train").sum()
parts = []
for _, g in ol.groupby("label"):
    parts += [g[g["split"] == "train"].sample(frac=frac, random_state=SEED), g[g["split"] != "train"]]
ol = pd.concat(parts)
pm = pd.read_csv(f"{OUT}/plantdoc_manifest_v1.csv")
pm = pm[pm["role"].isin(["train", "val", "cal", "ext_test", "holdout"])]
is_t = pm["species"] == "tomato"

A = pd.DataFrame({"source": "olazo", "rel_path": ol["path"], "label": 1, "group": "tomato",
                  "style": "lab", "split": ol["split"], "md5": ol["md5"]})
B = pd.DataFrame({"source": "plantdoc", "rel_path": pm["path"], "label": is_t.astype(int),
                  "group": np.where(is_t, "tomato", pm["species"]), "style": "field",
                  "split": pm["role"], "md5": pm["md5"]})
C = pd.DataFrame({"source": neg["source"], "rel_path": neg["rel_path"], "label": 0, "group": neg["group"],
                  "style": neg["style"], "split": neg["split"], "md5": neg["md5"]})
G = pd.concat([A, B, C], ignore_index=True)

conf = G.groupby("md5")["label"].nunique()
G = G[~G["md5"].isin(set(conf[conf > 1].index))]
prio = {"ext_test": 0, "holdout": 1, "test": 2, "cal": 3, "val": 4, "train": 5}
G = (G.assign(p=G["split"].map(prio)).sort_values(["p", "source", "rel_path"])
       .drop_duplicates("md5", keep="first").drop(columns="p").reset_index(drop=True))
assert G["md5"].is_unique and len(G) == EXPECTED["gate_rows"], len(G)
G.to_csv(f"{OUT}/gate_manifest_v1.csv", index=False)
print(pd.crosstab([G["label"], G["style"]], G["split"])[["train", "val", "cal", "test", "ext_test", "holdout"]])

Using Colab cache for faster access to the 'tomato-diseases' dataset.
Using Colab cache for faster access to the 'plantdisease' dataset.
Using Colab cache for faster access to the 'intel-image-classification' dataset.
Using Colab cache for faster access to the 'natural-images' dataset.


natural: 100%|██████████| 13798/13798 [01:33<00:00, 147.79it/s]


split        train   val   cal  test  ext_test  holdout
label style                                            
0     field    762    95    95     0        92      779
      lab     3238   463   463   463         0        0
      scene   1402   198   198   198         0        0
1     field    540    67    67     0        69        0
      lab     5500  1774  1775  4437         0        0


In [16]:
REBUILD = False
MIN_SIDE = 64
POS = ["Tomato Brown Spots", "Tomato blight leaf", "Tomato healthy", "Tomato leaf mosaic virus", "Tomato leaf yellow virus"]
NEG = ["Cassava Bacterial Blight", "Cassava Brown Leaf Spot", "Cassava Healthy", "Cassava Mosaic", "Cassava Root Rot"]
CLF_MAP = {"Tomato healthy": "healthy", "Tomato leaf mosaic virus": "mosaic_virus",
           "Tomato leaf yellow virus": "yellow_leaf_curl_virus"}

def build_fieldplant():
    import getpass
    from roboflow import Roboflow
    rf = Roboflow(api_key=getpass.getpass("Roboflow private API key (type here only): ").strip())
    ds = rf.workspace("plant-disease-detection").project("fieldplant").version(11).download("coco")
    fp = os.path.join(ds.location, "train")
    coco = json.load(open(f"{fp}/_annotations.coco.json"))
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    files = {i["id"]: i["file_name"] for i in coco["images"]}
    assert set(POS + NEG) <= set(cats.values())

    keep = [a for a in coco["annotations"] if cats[a["category_id"]] in set(POS + NEG)
            and min(a["bbox"][2], a["bbox"][3]) >= MIN_SIDE]
    ids = sorted({a["image_id"] for a in keep})
    np.random.RandomState(SEED).shuffle(ids)
    n = len(ids)
    split_of = {i: ("train" if k < 0.7 * n else "val" if k < 0.8 * n else "cal" if k < 0.9 * n else "test")
                for k, i in enumerate(ids)}

    shutil.rmtree(CROPS, ignore_errors=True)
    rows = []
    for a in tqdm(keep):
        cat = cats[a["category_id"]]
        try:
            with Image.open(os.path.join(fp, files[a["image_id"]])) as im:
                x, y, w, h = a["bbox"]
                crop = im.convert("RGB").crop((max(0, int(x)), max(0, int(y)),
                                               min(im.width, int(x + w)), min(im.height, int(y + h))))
            if min(crop.size) < MIN_SIDE:
                continue
            rel = f"{cat.replace(' ', '_')}/{a['image_id']}_{a['id']}.jpg"
            path = os.path.join(CROPS, rel)
            os.makedirs(os.path.dirname(path), exist_ok=True)
            crop.save(path, quality=95)
        except Exception:
            continue
        is_t = cat in POS
        rows.append(("fieldplant", rel, int(is_t), "tomato" if is_t else "cassava", "field",
                     split_of[a["image_id"]], md5_file(path), cat, CLF_MAP.get(cat), a["image_id"]))

    F = pd.DataFrame(rows, columns=["source", "rel_path", "label", "group", "style", "split",
                                    "md5", "category", "classifier_label", "image_id"])
    F = F.drop_duplicates("md5", keep="first").reset_index(drop=True)
    assert (F.groupby("image_id")["label"].nunique() == 1).all(), "a photo has both tomato and cassava crops"
    assert F.groupby("image_id")["split"].nunique().max() == 1, "a photo is split across sets"
    F.to_csv(f"{OUT}/fieldplant_manifest_v1.csv", index=False)
    shutil.make_archive(f"{BASE_DIR}/data/fieldplant_crops", "zip", CROPS)
    return F

have = os.path.exists(f"{OUT}/fieldplant_manifest_v1.csv") and os.path.exists(f"{BASE_DIR}/data/fieldplant_crops.zip")
if have and not REBUILD:
    print("FieldPlant crops already built; skipping (set REBUILD = True to redo).")
    F = pd.read_csv(f"{OUT}/fieldplant_manifest_v1.csv")
else:
    F = build_fieldplant()
print(pd.crosstab(F["category"], F["split"]).reindex(columns=["train", "val", "cal", "test"], fill_value=0).to_string())

Roboflow private API key (type here only): ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to FieldPlant-11 in coco:: 100%|██████████| 5160/5160 [00:48<00:00, 107.15it/s]
100%|██████████| 5409/5409 [10:48<00:00,  8.34it/s]


split                     train  val  cal  test
category                                       
Cassava Bacterial Blight     19    2    2     2
Cassava Brown Leaf Spot     144   26   16    25
Cassava Healthy             286   49   36    46
Cassava Mosaic             1317  194  216   168
Cassava Root Rot             51    9    7    10
Tomato Brown Spots         1428  195  158   186
Tomato blight leaf          275   38   53    38
Tomato healthy              193   26   26    34
Tomato leaf mosaic virus     11    2    3     3
Tomato leaf yellow virus     78   11   11    15


In [17]:
need = [f"{OUT}/{n}" for n in ("splits_v1.csv", "plantdoc_manifest_v1.csv", "gate_manifest_v1.csv",
                               "fieldplant_manifest_v1.csv")] + [f"{MODELS}/class_names.json"]
missing = [p for p in need if not os.path.exists(p)]
assert not missing, missing

fp = {
    "splits": fingerprint(pd.read_csv(f"{OUT}/splits_v1.csv"), ["md5", "split"]),
    "plantdoc": fingerprint(pd.read_csv(f"{OUT}/plantdoc_manifest_v1.csv"), ["md5", "role"]),
    "gate": fingerprint(pd.read_csv(f"{OUT}/gate_manifest_v1.csv"), ["md5", "split", "label"]),
    "fieldplant": fingerprint(pd.read_csv(f"{OUT}/fieldplant_manifest_v1.csv"), ["md5", "split", "label"]),
}
path = f"{OUT}/frozen_fingerprints.json"
saved = json.load(open(path)) if os.path.exists(path) else {}
print("Match with saved fingerprints:", {k: saved.get(k, "new") == v if k in saved else "new" for k, v in fp.items()})
json.dump({**fp, **saved}, open(path, "w"), indent=2)
print("Class order:", json.load(open(f"{MODELS}/class_names.json")))

Match with saved fingerprints: {'splits': 'new', 'plantdoc': 'new', 'gate': 'new', 'fieldplant': 'new'}
Class order: {'0': 'bacterial_spot', '1': 'early_blight', '2': 'healthy', '3': 'late_blight', '4': 'leaf_mold', '5': 'mosaic_virus', '6': 'septoria_leaf_spot', '7': 'target_spot', '8': 'twospotted_spider_mite', '9': 'yellow_leaf_curl_virus'}


In [18]:
roots = get_roots()
G0 = pd.read_csv(f"{OUT}/gate_manifest_v1.csv")
F = pd.read_csv(f"{OUT}/fieldplant_manifest_v1.csv")[["source", "rel_path", "label", "group", "style", "split", "md5"]]
assert not (set(F["md5"]) & set(G0["md5"])), "FieldPlant overlaps existing gate images"
G = pd.concat([G0, F], ignore_index=True)
G["file"] = [os.path.join(roots[s], r) for s, r in zip(G["source"], G["rel_path"])]
missing = [p for p in G["file"] if not os.path.exists(p)]
assert not missing, missing[:3]
assert G["md5"].is_unique

tr = G[G["split"] == "train"]
print("Rows:", len(G), "|", G["split"].value_counts().to_dict())
print("\nTrain images by source (columns: 0 = not tomato, 1 = tomato)")
print(pd.crosstab(tr["source"] + "_" + tr["style"], tr["label"]).to_string())
print("\nTrain totals:", tr["label"].value_counts().to_dict())

Using Colab cache for faster access to the 'tomato-diseases' dataset.
Using Colab cache for faster access to the 'plantdisease' dataset.
Using Colab cache for faster access to the 'intel-image-classification' dataset.
Using Colab cache for faster access to the 'natural-images' dataset.
Rows: 28084 | {'train': 15244, 'test': 5625, 'val': 3149, 'cal': 3126, 'holdout': 779, 'ext_test': 161}

Train images by source (columns: 0 = not tomato, 1 = tomato)
label                0     1
row_0                       
emmarex_lab       3238     0
fieldplant_field  1817  1985
intel_scene        690     0
natural_scene      712     0
olazo_lab            0  5500
plantdoc_field     762   540

Train totals: {1: 8025, 0: 7219}


In [19]:
os.makedirs(f"{MODELS}/archive", exist_ok=True)
for name in ("agrilens_pro_gate.keras", "agrilens_pro_gate_finetuned.keras",
             "agrilens_pro_gate_with_fieldplant.keras", "gate_threshold.json"):
    if os.path.exists(f"{MODELS}/{name}"):
        shutil.move(f"{MODELS}/{name}", f"{MODELS}/archive/{name}")

print("GPUs:", tf.config.list_physical_devices("GPU"))
train_df, val_df = G[G["split"] == "train"], G[G["split"] == "val"]
train_ds, val_ds = make_ds(train_df, train=True), make_ds(val_df)
n0, n1 = int((train_df["label"] == 0).sum()), int((train_df["label"] == 1).sum())
cw = {0: len(train_df) / (2 * n0), 1: len(train_df) / (2 * n1)}
print("Class weights:", {k: round(v, 3) for k, v in cw.items()})

aug = keras.Sequential([
    keras.layers.RandomFlip("horizontal"), keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom(0.15), keras.layers.RandomContrast(0.2),
    keras.layers.RandomBrightness(0.2)], name="augmentation")
base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
base.trainable = False
inp = keras.Input(shape=(IMG, IMG, 3))
x = aug(inp)
x = keras.layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_rescale")(x)
x = base(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)
out = keras.layers.Dense(1, activation="sigmoid", name="p_tomato_leaf")(x)
gate = keras.Model(inp, out, name="agrilens_pro_gate")

def run_stage(lr, epochs, patience, ckpt):
    gate.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                 metrics=["accuracy", keras.metrics.AUC(name="auc")])
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7),
           keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True)]
    h = gate.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=cw, callbacks=cbs)
    b = int(np.argmin(h.history["val_loss"]))
    print(f"-> best epoch {b + 1}: val_loss {h.history['val_loss'][b]:.4f} | "
          f"val_acc {h.history['val_accuracy'][b]:.4f} | val_auc {h.history['val_auc'][b]:.4f}")
    return {k: [float(v) for v in vals] for k, vals in h.history.items()}

print("\n=== Stage 1: head only ===")
h1 = run_stage(1e-3, 12, 4, f"{MODELS}/archive/gate_stage1.keras")

base.trainable = True
for l in base.layers[:-40]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, keras.layers.BatchNormalization):
        l.trainable = False
print("\n=== Stage 2: fine-tune last 40 layers ===")
final = f"{MODELS}/agrilens_pro_gate.keras"
h2 = run_stage(1e-5, 10, 3, final)

json.dump({"stage1": h1, "stage2": h2}, open(f"{OUT}/gate_training_history.json", "w"))
print("Saved:", final, f"({os.path.getsize(final) / 1e6:.1f} MB)")
from google.colab import files as gfiles
gfiles.download(final)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Class weights: {0: 1.056, 1: 0.95}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== Stage 1: head only ===
Epoch 1/12
477/477 ━━━━━━━━━━━━━━━━━━━━ 157s 298ms/step - accuracy: 0.9078 - auc: 0.9674 - loss: 0.2306 - val_accuracy: 0.9597 - val_auc: 0.9901 - val_loss: 0.1123 - learning_rate: 0.0010
Epoch 2/12
477/477 ━━━━━━━━━━━━━━━━━━━━ 137s 287ms/step - accuracy: 0.9420 - auc: 0.9861 - loss: 0.1491 - val_accuracy: 0.9625 - val_auc: 0.9915 - val_loss: 0.1012 - learning_rate: 0.0010
Epoch 3/12
477/477 ━━━━━━━━━━━━━━━━━━━━ 127s 266ms/step - accuracy: 0.9448 - auc: 0.9878 - loss: 0.1390 - val_accuracy: 0.9676 - val_auc: 0.9945 - val_loss: 0.0913 - learning_rate: 0.0010
Epoch 4/12
477/477 ━━━━━━━━━━━━━━━━━━━━ 119s 249ms/step - accuracy: 0.9539 - auc: 0.9910 - loss: 0.1188 - val_accuracy: 0.9714 - val_auc: 0.9937 - val_loss: 0.0858 - learning_rate: 0.0010
Epoch 5/12
477/477 ━━━━━━━━━━━━━━━━━━━━ 122s 255ms/step - accur

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
P, thr = evaluate_gate(f"{MODELS}/agrilens_pro_gate.keras", G, "final", save_threshold=True)

Cal split accept rates (tomato = recall, others = false accepts)
      bell_pepper_lab  cassava_fp  field_other_species  object  potato_lab  scene  tomato_field  tomato_field_fp  tomato_lab
0.50              0.0       0.004                0.126     0.0       0.009    0.0         0.746            0.984       0.995
0.60              0.0       0.000                0.074     0.0       0.009    0.0         0.657            0.976       0.994
0.70              0.0       0.000                0.053     0.0       0.005    0.0         0.522            0.976       0.993
0.80              0.0       0.000                0.042     0.0       0.005    0.0         0.418            0.968       0.988
0.90              0.0       0.000                0.011     0.0       0.005    0.0         0.224            0.944       0.979
0.95              0.0       0.000                0.011     0.0       0.000    0.0         0.104            0.904       0.965
0.98              0.0       0.000                0.000     0

In [24]:
# Cell 14: Build 10-class Tomato Disease Classifier Manifest
roots = get_roots()
class_map = {v: int(k) for k, v in json.load(open(f"{MODELS}/class_names.json")).items()}
num_classes = len(class_map)

# 1. Olazo lab dataset (all 10 classes)
ol = pd.read_csv(f"{OUT}/splits_v1.csv")
df_ol = pd.DataFrame({
    "source": "olazo",
    "style": "lab",
    "rel_path": ol["path"],
    "class_name": ol["label"],
    "split": ol["split"],
    "md5": ol["md5"]
})

# 2. PlantDoc field tomato dataset (7 mapped classes)
pm = pd.read_csv(f"{OUT}/plantdoc_manifest_v1.csv")
pm = pm[(pm["species"] == "tomato") & (pm["classifier_label"].notna()) & (pm["role"].isin(["train", "val", "cal", "ext_test"]))].copy()
df_pm = pd.DataFrame({
    "source": "plantdoc",
    "style": "field",
    "rel_path": pm["path"],
    "class_name": pm["classifier_label"],
    "split": pm["role"],
    "md5": pm["md5"]
})

# 3. FieldPlant field tomato crops (3 mapped classes: healthy, mosaic_virus, yellow_leaf_curl_virus)
fp_df = pd.read_csv(f"{OUT}/fieldplant_manifest_v1.csv")
fp_df = fp_df[(fp_df["label"] == 1) & (fp_df["classifier_label"].notna())].copy()
df_fp = pd.DataFrame({
    "source": "fieldplant",
    "style": "field",
    "rel_path": fp_df["rel_path"],
    "class_name": fp_df["classifier_label"],
    "split": fp_df["split"],
    "md5": fp_df["md5"]
})

C = pd.concat([df_ol, df_pm, df_fp], ignore_index=True)
C["label"] = C["class_name"].map(class_map)
assert C["label"].notna().all(), "Unmapped class names found"
C["label"] = C["label"].astype(int)
C["file"] = [os.path.join(roots[s], r) for s, r in zip(C["source"], C["rel_path"])]

# Verify files exist and no MD5 overlap across splits
missing = [p for p in C["file"] if not os.path.exists(p)]
assert not missing, f"Missing files: {missing[:3]}"
C = C.drop_duplicates("md5", keep="first").reset_index(drop=True)

splits_md5 = {s: set(g["md5"]) for s, g in C.groupby("split")}
assert all(not (splits_md5[a] & splits_md5[b]) for a in splits_md5 for b in splits_md5 if a < b), "Data leakage across splits!"

C.to_csv(f"{OUT}/classifier_manifest_v1.csv", index=False)
print("Total classifier images:", len(C))
print("Split totals:", C["split"].value_counts().to_dict())
print("\nClass distribution by split:")
print(pd.crosstab(C["class_name"], C["split"])[["train", "val", "cal", "test", "ext_test"]].to_string())


Using Colab cache for faster access to the 'tomato-diseases' dataset.
Using Colab cache for faster access to the 'plantdisease' dataset.
Using Colab cache for faster access to the 'intel-image-classification' dataset.
Using Colab cache for faster access to the 'natural-images' dataset.
Total classifier images: 23269
Split totals: {'train': 14970, 'test': 4489, 'cal': 1876, 'val': 1874, 'ext_test': 60}

Class distribution by split:
split                   train  val  cal  test  ext_test
class_name                                             
bacterial_spot           1424  178  178   444         8
early_blight             1804  226  226   444         9
healthy                  1695  214  214   477         0
late_blight              1750  219  219   442        10
leaf_mold                1237  154  154   444         6
mosaic_virus             1005  126  126   447        10
septoria_leaf_spot       1172  146  147   444        11
target_spot               888  111  111   444         0
twosp

In [26]:
# Train 10-class Tomato Disease Classifier
os.makedirs(f"{MODELS}/archive", exist_ok=True)
if os.path.exists(f"{MODELS}/agrilens_pro_classifier.keras"):
    shutil.move(f"{MODELS}/agrilens_pro_classifier.keras", f"{MODELS}/archive/agrilens_pro_classifier.keras")

def _load_clf(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG), method="bilinear")
    return tf.cast(img, tf.float32), tf.cast(label, tf.int32)

def make_clf_ds(df, train=False, bs=32):
    ds = tf.data.Dataset.from_tensor_slices((df["file"].tolist(), df["label"].astype("int32").values))
    if train:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(_load_clf, num_parallel_calls=tf.data.AUTOTUNE).batch(bs).prefetch(tf.data.AUTOTUNE)

tr_clf = C[C["split"] == "train"]
va_clf = C[C["split"] == "val"]
train_clf_ds = make_clf_ds(tr_clf, train=True)
val_clf_ds = make_clf_ds(va_clf, train=False)

# Balanced class weights for the 10 classes
counts = tr_clf["label"].value_counts().to_dict()
clf_cw = {i: len(tr_clf) / (num_classes * counts[i]) for i in range(num_classes)}
print("Classifier class weights:", {k: round(v, 3) for k, v in sorted(clf_cw.items())})

aug_clf = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomFlip("vertical"),
    keras.layers.RandomRotation(0.2),
    keras.layers.RandomZoom(0.15),
    keras.layers.RandomContrast(0.2),
    keras.layers.RandomBrightness(0.2)
], name="clf_augmentation")

base_clf = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
base_clf.trainable = False

inp = keras.Input(shape=(IMG, IMG, 3))
x = aug_clf(inp)
x = keras.layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_rescale")(x)
x = base_clf(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.25)(x)
out = keras.layers.Dense(num_classes, activation="softmax", name="disease_probs")(x)
clf_model = keras.Model(inp, out, name="agrilens_pro_classifier")

def run_clf_stage(lr, epochs, patience, ckpt):
    clf_model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    cbs = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True)
    ]
    h = clf_model.fit(train_clf_ds, validation_data=val_clf_ds, epochs=epochs, class_weight=clf_cw, callbacks=cbs)
    b = int(np.argmin(h.history["val_loss"]))
    print(f"-> best epoch {b + 1}: val_loss {h.history['val_loss'][b]:.4f} | val_acc {h.history['val_accuracy'][b]:.4f}")
    return {k: [float(v) for v in vals] for k, vals in h.history.items()}

print("\n=== Classifier Stage 1: head only ===")
ch1 = run_clf_stage(1e-3, 12, 4, f"{MODELS}/archive/clf_stage1.keras")

base_clf.trainable = True
for l in base_clf.layers[:-50]:
    l.trainable = False
for l in base_clf.layers:
    if isinstance(l, keras.layers.BatchNormalization):
        l.trainable = False

print("\n=== Classifier Stage 2: fine-tune last 50 layers ===")
clf_final = f"{MODELS}/agrilens_pro_classifier.keras"
ch2 = run_clf_stage(2e-5, 15, 4, clf_final)

json.dump({"stage1": ch1, "stage2": ch2}, open(f"{OUT}/classifier_training_history.json", "w"))
print("Saved:", clf_final, f"({os.path.getsize(clf_final) / 1e6:.1f} MB)")


Classifier class weights: {0: 1.051, 1: 0.83, 2: 0.883, 3: 0.855, 4: 1.21, 5: 1.49, 6: 1.277, 7: 1.686, 8: 1.518, 9: 0.498}

=== Classifier Stage 1: head only ===
Epoch 1/12
468/468 ━━━━━━━━━━━━━━━━━━━━ 71s 134ms/step - accuracy: 0.6600 - loss: 1.0523 - val_accuracy: 0.8004 - val_loss: 0.6059 - learning_rate: 0.0010
Epoch 2/12
468/468 ━━━━━━━━━━━━━━━━━━━━ 36s 76ms/step - accuracy: 0.7577 - loss: 0.7375 - val_accuracy: 0.8084 - val_loss: 0.5909 - learning_rate: 0.0010
Epoch 3/12
468/468 ━━━━━━━━━━━━━━━━━━━━ 38s 82ms/step - accuracy: 0.7841 - loss: 0.6622 - val_accuracy: 0.8170 - val_loss: 0.5467 - learning_rate: 0.0010
Epoch 4/12
468/468 ━━━━━━━━━━━━━━━━━━━━ 36s 77ms/step - accuracy: 0.7894 - loss: 0.6377 - val_accuracy: 0.8383 - val_loss: 0.4939 - learning_rate: 0.0010
Epoch 5/12
468/468 ━━━━━━━━━━━━━━━━━━━━ 37s 79ms/step - accuracy: 0.7986 - loss: 0.6081 - val_accuracy: 0.8324 - val_loss: 0.4766 - learning_rate: 0.0010
Epoch 6/12
468/468 ━━━━━━━━━━━━━━━━━━━━ 39s 75ms/step - accuracy: 

In [27]:
# Classifier Calibration & Evaluation (with Wilson 95% CIs)
from sklearn.metrics import classification_report

clf_m = tf.keras.models.load_model(f"{MODELS}/agrilens_pro_classifier.keras")
idx_to_cls = {int(k): v for k, v in json.load(open(f"{MODELS}/class_names.json")).items()}

parts = []
for sp in ("cal", "test", "ext_test"):
    d = C[C["split"] == sp].copy()
    if len(d):
        probs = clf_m.predict(make_clf_ds(d, bs=64), verbose=0)
        d["pred"] = probs.argmax(axis=1)
        d["pred_name"] = d["pred"].map(idx_to_cls)
        d["conf"] = probs.max(axis=1)
        d["correct"] = (d["pred"] == d["label"]).astype(int)
        parts.append(d)

CP = pd.concat(parts, ignore_index=True)

# 1. Calibrate abstain threshold on `cal` split (target >= 95% selective accuracy)
cal = CP[CP["split"] == "cal"]
grid = [0.0, 0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
cal_rows = []
for t in grid:
    sub = cal[cal["conf"] >= t]
    cov = len(sub) / len(cal)
    acc = sub["correct"].mean() if len(sub) else 0.0
    lo, hi = wilson(int(sub["correct"].sum()), len(sub)) if len(sub) else (0.0, 0.0)
    cal_rows.append((t, len(sub), round(cov, 3), round(acc, 3), round(max(lo, 0.0), 3), round(hi, 3)))

cal_tab = pd.DataFrame(cal_rows, columns=["thr", "kept", "coverage", "sel_acc", "ci_lo", "ci_hi"])
print("Cal split confidence sweep (selective accuracy vs coverage):")
print(cal_tab.to_string(index=False))

ok = cal_tab[cal_tab["sel_acc"] >= 0.95]
clf_thr = float(ok.iloc[0]["thr"]) if len(ok) else 0.80
print(f"\nChosen classifier threshold: {clf_thr}")

json.dump({
    "threshold": clf_thr,
    "rule": "accept diagnosis if max_prob >= threshold else abstain (uncertain / retake)",
    "chosen_on": "cal split, smallest threshold achieving >= 95% selective accuracy"
}, open(f"{MODELS}/classifier_threshold.json", "w"), indent=2)

# 2. Report overall & selective accuracy by split and source
def clf_report_group(df, t):
    rows = []
    for (sp, src), x in df.groupby(["split", "source"]):
        n, k_raw = len(x), int(x["correct"].sum())
        lo_r, hi_r = wilson(k_raw, n)
        sub = x[x["conf"] >= t]
        n_s, k_s = len(sub), int(sub["correct"].sum())
        lo_s, hi_s = wilson(k_s, n_s) if n_s else (0.0, 0.0)
        rows.append((
            sp, src, n,
            round(k_raw / n, 3), f"[{max(lo_r,0):.3f}, {hi_r:.3f}]",
            round(n_s / n, 3),
            round(k_s / n_s, 3) if n_s else np.nan,
            f"[{max(lo_s,0):.3f}, {hi_s:.3f}]" if n_s else "n/a"
        ))
    return pd.DataFrame(rows, columns=["split", "source", "n", "raw_acc", "raw_95ci", "cov@thr", "sel_acc", "sel_95ci"])

print("\n--- Overall & Selective Accuracy by Split/Source ---")
print(clf_report_group(CP, clf_thr).to_string(index=False))

# 3. Per-class test performance (Olazo + FieldPlant combined test)
te = CP[CP["split"] == "test"]
cls_rows = []
for c_idx in range(num_classes):
    cname = idx_to_cls[c_idx]
    x = te[te["label"] == c_idx]
    n, k = len(x), int(x["correct"].sum())
    lo, hi = wilson(k, n)
    sub = x[x["conf"] >= clf_thr]
    sel_acc = round(sub["correct"].mean(), 3) if len(sub) else np.nan
    cls_rows.append((cname, n, round(k / n, 3), f"[{max(lo,0):.3f}, {hi:.3f}]", round(len(sub) / n, 3), sel_acc))

print("\n--- Per-Class Test Performance ---")
print(pd.DataFrame(cls_rows, columns=["class", "n", "raw_acc", "raw_95ci", "cov@thr", "sel_acc"]).to_string(index=False))

CP.drop(columns=["file"], errors="ignore").to_csv(f"{OUT}/classifier_predictions_final.csv", index=False)


Cal split confidence sweep (selective accuracy vs coverage):
 thr  kept  coverage  sel_acc  ci_lo  ci_hi
0.00  1876     1.000    0.929  0.916  0.939
0.50  1824     0.972    0.943  0.931  0.953
0.60  1758     0.937    0.955  0.944  0.964
0.70  1702     0.907    0.967  0.958  0.975
0.75  1667     0.889    0.972  0.963  0.979
0.80  1632     0.870    0.977  0.969  0.984
0.85  1597     0.851    0.980  0.972  0.986
0.90  1536     0.819    0.985  0.978  0.990
0.95  1410     0.752    0.993  0.987  0.996

Chosen classifier threshold: 0.6

--- Overall & Selective Accuracy by Split/Source ---
   split     source    n  raw_acc       raw_95ci  cov@thr  sel_acc       sel_95ci
     cal fieldplant   40    0.875 [0.739, 0.945]    0.850    0.941 [0.809, 0.984]
     cal      olazo 1775    0.946 [0.935, 0.956]    0.959    0.960 [0.950, 0.968]
     cal   plantdoc   61    0.443 [0.325, 0.567]    0.344    0.571 [0.365, 0.755]
ext_test   plantdoc   60    0.333 [0.227, 0.459]    0.350    0.571 [0.365, 0.755]
 

In [28]:
# Cell 17: End-to-End Two-Stage Evaluation & Standalone Inference Module
gate_m = tf.keras.models.load_model(f"{MODELS}/agrilens_pro_gate.keras")
clf_m = tf.keras.models.load_model(f"{MODELS}/agrilens_pro_classifier.keras")
gate_thr = json.load(open(f"{MODELS}/gate_threshold.json"))["threshold"]
clf_thr = json.load(open(f"{MODELS}/classifier_threshold.json"))["threshold"]
idx_to_cls = {int(k): v for k, v in json.load(open(f"{MODELS}/class_names.json")).items()}

# 1. End-to-End Evaluation on Gate + Classifier splits
# Merge true disease labels onto G where available
clf_labels = C[["md5", "class_name", "label"]].rename(columns={"label": "disease_label"})
E = G[G["split"].isin(["test", "ext_test", "holdout"])].merge(clf_labels, on="md5", how="left")
E["grp"] = E.apply(gate_group, axis=1)

# Run Stage 1 Gate
E["p_gate"] = gate_m.predict(make_ds(E, bs=64), verbose=0).ravel()

# Run Stage 2 Classifier on images passing Gate
pass_mask = E["p_gate"] >= gate_thr
E["p_clf"] = np.nan
E["pred_idx"] = -1
E["pred_class"] = None

if pass_mask.any():
    probs = clf_m.predict(make_ds(E[pass_mask], bs=64), verbose=0)
    E.loc[pass_mask, "p_clf"] = probs.max(axis=1)
    E.loc[pass_mask, "pred_idx"] = probs.argmax(axis=1)
    E.loc[pass_mask, "pred_class"] = [idx_to_cls[i] for i in probs.argmax(axis=1)]

def route_decision(r):
    if r["p_gate"] < gate_thr:
        return "reject_non_tomato"
    if r["p_clf"] < clf_thr:
        return "abstain_uncertain"
    return "accept_diagnosis"

E["decision"] = E.apply(route_decision, axis=1)
E["diag_correct"] = np.where(
    (E["decision"] == "accept_diagnosis") & (E["disease_label"].notna()),
    (E["pred_idx"] == E["disease_label"]).astype(float),
    np.nan
)

rows = []
for (sp, grp), x in E.groupby(["split", "grp"]):
    n = len(x)
    rej = (x["decision"] == "reject_non_tomato").mean()
    abt = (x["decision"] == "abstain_uncertain").mean()
    acc = (x["decision"] == "accept_diagnosis").mean()
    diagnosed = x[x["diag_correct"].notna()]
    acc_diag = round(diagnosed["diag_correct"].mean(), 3) if len(diagnosed) else np.nan
    rows.append((sp, grp, n, round(rej, 3), round(abt, 3), round(acc, 3), len(diagnosed), acc_diag))

e2e_df = pd.DataFrame(rows, columns=["split", "group", "n", "reject", "abstain", "accept", "n_eval", "diag_acc"])
print(f"End-to-End Two-Stage Pipeline (gate_thr={gate_thr}, clf_thr={clf_thr}):")
print(e2e_df.to_string(index=False))
E.drop(columns=["file"], errors="ignore").to_csv(f"{OUT}/end_to_end_eval_final.csv", index=False)

# 2. Save standalone deployment inference script
infer_code = '''import json
import numpy as np
import tensorflow as tf

class AgrilensPipeline:
    def __init__(self, models_dir="/content/drive/MyDrive/Agrilens_pro/models", img_size=224):
        self.img_size = img_size
        self.gate = tf.keras.models.load_model(f"{models_dir}/agrilens_pro_gate.keras")
        self.clf = tf.keras.models.load_model(f"{models_dir}/agrilens_pro_classifier.keras")
        self.gate_thr = json.load(open(f"{models_dir}/gate_threshold.json"))["threshold"]
        self.clf_thr = json.load(open(f"{models_dir}/classifier_threshold.json"))["threshold"]
        self.class_names = {int(k): v for k, v in json.load(open(f"{models_dir}/class_names.json")).items()}

    def _preprocess(self, image_path):
        raw = tf.io.read_file(image_path)
        img = tf.io.decode_image(raw, channels=3, expand_animations=False)
        img = tf.image.resize(img, (self.img_size, self.img_size), method="bilinear")
        return tf.expand_dims(tf.cast(img, tf.float32), axis=0)

    def predict(self, image_path):
        x = self._preprocess(image_path)
        p_tomato = float(self.gate.predict(x, verbose=0)[0][0])
        if p_tomato < self.gate_thr:
            return {
                "status": "rejected",
                "reason": "not_a_tomato_leaf",
                "p_tomato": round(p_tomato, 4),
                "diagnosis": None,
                "confidence": None
            }
        probs = self.clf.predict(x, verbose=0)[0]
        top_idx = int(np.argmax(probs))
        conf = float(probs[top_idx])
        if conf < self.clf_thr:
            return {
                "status": "retake",
                "reason": "low_confidence",
                "p_tomato": round(p_tomato, 4),
                "diagnosis": self.class_names[top_idx],
                "confidence": round(conf, 4)
            }
        return {
            "status": "accepted",
            "reason": "ok",
            "p_tomato": round(p_tomato, 4),
            "diagnosis": self.class_names[top_idx],
            "confidence": round(conf, 4)
        }
'''
with open(f"{BASE_DIR}/code/agrilens_inference.py", "w") as f:
    f.write(infer_code)

# Smoke test the standalone pipeline on 1 tomato leaf and 1 non-tomato image
from agrilens_inference import AgrilensPipeline
pipe = AgrilensPipeline(MODELS, IMG)
sample_pos = E[E["grp"] == "tomato_lab"].iloc[0]["file"]
sample_neg = E[E["grp"] == "bell_pepper_lab"].iloc[0]["file"]
print("\nSmoke test (tomato leaf):", pipe.predict(sample_pos))
print("Smoke test (non-tomato):", pipe.predict(sample_neg))


End-to-End Two-Stage Pipeline (gate_thr=0.8, clf_thr=0.6):
   split               group    n  reject  abstain  accept  n_eval  diag_acc
ext_test field_other_species   92   0.924    0.033   0.043       0       NaN
ext_test        tomato_field   69   0.536    0.232   0.232      14     0.571
 holdout        unseen_apple  273   0.945    0.022   0.033       0       NaN
 holdout         unseen_corn  373   0.936    0.035   0.029       0       NaN
 holdout        unseen_grape  133   0.902    0.060   0.038       0       NaN
    test     bell_pepper_lab  248   1.000    0.000   0.000       0       NaN
    test          cassava_fp  251   0.992    0.004   0.004       0       NaN
    test              object   96   1.000    0.000   0.000       0       NaN
    test          potato_lab  215   0.991    0.005   0.005       0       NaN
    test               scene  102   1.000    0.000   0.000       0       NaN
    test     tomato_field_fp  276   0.047    0.373   0.580      43     0.930
    test         

In [29]:
from google.colab import files

files.download(f"{MODELS}/agrilens_pro_gate.keras")
files.download(f"{MODELS}/agrilens_pro_classifier.keras")
files.download(f"{MODELS}/class_names.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
files.download(f"{MODELS}/agrilens_pro_classifier.keras")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>